# Optimizing Model Parameters

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

导入数据，定义网络

# 超参数
这里一般用到三个超参数
- Epochs：对数据集进行迭代的次数
- Batch Size：参数更新之前通过网络传播的数据样本的数量
- Learning Rate：每个批次/时期更新模型参数的程度。较小的值会导致学习速度较慢，而较大的值可能会导致训练期间出现不可预测的行为

In [ ]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

# 优化循环
一旦设定好超参数，就可以用通过优化循环来训练和优化模型。优化循环的每次迭代称为一个纪元 epoch。

每个时代由两个主要部分组成：
- 训练循环——遍历训练数据集，尝试收敛到最优参数。
- 验证/测试循环——遍历测试数据集，检查模型性能是否在提升。

## 损失函数
常用损失函数
- nn.MSELoss：均方误差，常用于回归任务
- nn.NLLLoss：负对数似然，常用于分类任务
- nn.CrossEntropyLoss：结合了 nn.LogSoftmax 和 nn.NLLLoss

In [ ]:
# Initialize the loss function
loss_fn = nn.CrossEntropyLoss()

## 优化器
在训练循环中，优化分三个步骤进行：
- 调用optimizer.zero_grad()重置模型参数的梯度。默认情况下渐变相加，为了防止重复计算，在每次迭代前明确地将它们归零
- 通过调用 loss.backward() 反向传播预测损失。PyTorch 存储损失相较于每个优化参数的梯度
- 一旦得到了梯度，就调用optimizer.step()来通过后向传递中收集的梯度来调整参数

In [ ]:
# 以训练模型的参数和学习率来初始化一个随机梯度下降优化器
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# 完整流程

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        # 更新权重，梯度下降法更新即 weight = weight - learning_rate * gradient
        optimizer.step()
        # 梯度清零
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

开始训练

In [ ]:
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")